# Kilonova (KNe) detection-efficiency metric - MAF implementation and demo


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-10
- Context: SCOC (Survey Cadence Optimization Committee). Notebooks 01/01b/02/03 in this series cover the **DESC** Task Force metrics proper (3x2pt, WL, SN). This notebook, **04**, covers a related SCOC-tracked transient science case that in the official `rubin_sim.maf` batch actually lives under the **"Variables/Transients"** group rather than the "Cosmology"/DESC group: the detection efficiency of **kilonovae (KNe)**, the electromagnetic counterparts of binary-neutron-star (and neutron-star-black-hole) mergers such as GW170817. It is kept in this series because it follows the exact same cadence-vs-single-metric workflow, and it is directly relevant to SCOC's multi-messenger / time-domain science case.
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Reproduces the official `rubin_sim.maf.batches.science_radar_batch` "Variables/Transients / KNe" subgroup, so results are directly comparable to the standard show_maf pages for this OpSim run.


## Notebook overview

Unlike every metric used so far in this series, the kilonova metric does **not** use a `HealpixSlicer`. Instead, `rubin_sim.maf.maf_contrib.kne_metrics` implements a small **Monte Carlo population-synthesis** pipeline:

1. **`get_kne_filename`** selects one or more kilonova light-curve template files from a pre-computed grid of radiative-transfer models (`POSSIS`, Bulla 2019), stored as `$RUBIN_SIM_DATA_DIR/maf/bns/*.dat`. Each file corresponds to a binary-neutron-star (`nsns`) or neutron-star-black-hole (`nsbh`) merger model with specific dynamical ejecta mass (`mej_dyn`), wind ejecta mass (`mej_wind`), lanthanide-rich opening angle (`phi`), and observer viewing angle (`theta`).
2. **`generate_kn_pop_slicer`** builds a `UserPointsSlicer`: it draws `n_events` fake kilonovae with random sky positions (uniform on the sphere), random peak times over the survey, a random light-curve template (from the file list), and a random luminosity distance (drawn to approximate a uniform-in-volume distribution between `d_min` and `d_max`). Each event is one "slice point" instead of one Healpix pixel.
3. **`KNePopMetric`** then interpolates the chosen template light curve at the actual visit times and bands observed at that sky position (using the real cadence, filters and 5-sigma depths), applies Galactic dust extinction (`DustMap`) and the distance modulus, and evaluates several independent, published **detection criteria** on the resulting simulated observations:
   - `multi_detect`: detected (S/N above the visit's 5-sigma limit) at least `pts_needed` times within 30 days of peak;
   - `ztfrest_simple` (+ `_red` / `_blue` band-restricted variants): a simplified version of the ZTFReST fast-transient rise/fade-rate selection (Andreoni & Coughlin et al. 2021);
   - `multi_color_detect`: detected in at least two different filters;
   - `red_color_detect` / `blue_color_detect`: at least 4 detections in the red (`izy`) or blue (`ugr`) bands.

Because `KNePopMetric.run()` returns one boolean per criterion, MAF's reduce-function mechanism (as in notebook 03) automatically splits the output into **seven** separate per-event bundles - one per detection criterion. Each is then reduced with the standard `lightcurve_summary()` statistics: total number detected, total events landing in the observed footprint, total events over the whole sky, and the two **detection efficiencies** (fraction detected of the observed footprint, and of the full injected population) - these efficiencies are the headline numbers for this metric.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/maf_contrib/kne_metrics.py` (`get_kne_filename`, `KnLc`, `KNePopMetric`, `generate_kn_pop_slicer`)
  - `rubin_sim/maf/batches/common.py` (`lightcurve_summary`, the standard summary-statistics set for population/light-curve metrics)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Variables/Transients" batch definition, "KNe" subgroup, function `science_radar_batch`)
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/


## Practical note before running

`KNePopMetric` needs the Bulla kilonova model grid, shipped as `.dat` files under `$RUBIN_SIM_DATA_DIR/maf/bns/`; if `get_kne_filename` returns an empty list, run `rs_download_data --dirs maf` first. The official batch injects **500,000** events - each is a cheap light-curve interpolation, but querying visits for half a million random sky positions still takes a few minutes on a laptop. This notebook uses the same `n_events=500000` as the official batch for direct comparability; reduce it (e.g. to a few thousand) for a quick, lower-precision test run.

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.batches.common import lightcurve_summary
from rubin_sim.maf.maf_contrib.kne_metrics import (
    get_kne_filename,
    KnLc,
    KNePopMetric,
    generate_kn_pop_slicer,
)

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

Same OpSim file and output-directory convention as the rest of the series, with a dedicated `NB_TAG`.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "KNE"
data_dir = f"data_04_{NB_TAG}"
figs_dir = f"figs_04_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF classes used for the kilonova population metric

In [ ]:
print(inspect.getdoc(get_kne_filename))
print("-" * 80)
print(inspect.getdoc(generate_kn_pop_slicer))
print("-" * 80)
print("KNePopMetric reduce functions (one output map each):")
print(list(KNePopMetric().reduce_funcs.keys()))
print("-" * 80)
print("lightcurve_summary() statistics applied to each of the maps above:")
print([m.name for m in lightcurve_summary()])

## 4. Configuration (matching the official `science_radar_batch` "KNe" subgroup)

Two model configurations are run, exactly as in the official batch:
- **`single model`**: a single GW170817-like template (`mej_dyn=0.005`, `mej_wind=0.050`, `phi=30`, `theta=25.8`), i.e. a specific, canonical kilonova viewed from a specific inclination;
- **`all models`**: the *entire* Bulla model grid (all ejecta masses, opening angles and viewing angles available), giving a population-averaged detection efficiency.

Both use `d_min=10`, `d_max=600` Mpc and are restricted to the non-DD footprint.

In [ ]:
n_events = 500_000
sqlconstraint = "scheduler_note not like 'DD%'"
d_min, d_max = 10, 600

# Single, GW170817-like model
single_inj_params = [{"mej_dyn": 0.005, "mej_wind": 0.050, "phi": 30, "theta": 25.8}]
single_filenames = get_kne_filename(single_inj_params)
print(f"'single model': {len(single_filenames)} matching template file(s)")

# Entire model grid
all_filenames = get_kne_filename(None)
print(f"'all models': {len(all_filenames)} template files in the Bulla grid")

## 5. Running the metric

In [ ]:
def run_kne_bundle(filenames, metric_name, info_label, n_events, d_min, d_max, out_dir):
    """Build the KNe population slicer + KNePopMetric bundle and run it."""
    kn_slicer = generate_kn_pop_slicer(n_events=n_events, n_files=len(filenames), d_min=d_min, d_max=d_max)
    metric = KNePopMetric(output_lc=False, file_list=filenames, metric_name=metric_name)
    bundle = mb.MetricBundle(
        metric,
        kn_slicer,
        sqlconstraint,
        run_name=run_name,
        info_label=info_label,
        summary_metrics=lightcurve_summary(),
    )
    bd = mb.make_bundles_dict_from_list([bundle])
    bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=out_dir, results_db=resultsDb)
    bgroup.run_all()
    return bgroup

In [ ]:
t0 = time.time()
bgroup_single = run_kne_bundle(
    single_filenames, "KNePopMetric_single", "single model", n_events, d_min, d_max, data_dir
)
print(f"single-model run done in {time.time() - t0:.1f} s")

In [ ]:
t0 = time.time()
bgroup_all = run_kne_bundle(all_filenames, "KNePopMetric_all", "all models", n_events, d_min, d_max, data_dir)
print(f"all-models run done in {time.time() - t0:.1f} s")

## 6. Results: detection efficiency by criterion

In [ ]:
criteria = [
    "multi_detect",
    "ztfrest_simple",
    "ztfrest_simple_red",
    "ztfrest_simple_blue",
    "multi_color_detect",
    "red_color_detect",
    "blue_color_detect",
]


def summary_table(bgroup, base_metric_name):
    rows = {}
    for c in criteria:
        key = f"{base_metric_name}_{c}"
        sv = bgroup.bundle_dict[key].summary_values
        rows[c] = sv
    return pd.DataFrame(rows).T


single_table = summary_table(bgroup_single, "KNePopMetric_single")
all_table = summary_table(bgroup_all, "KNePopMetric_all")

print("Single, GW170817-like model:")
display(single_table)
print("\nEntire Bulla model grid:")
display(all_table)

In [ ]:
single_table.to_csv(os.path.join(data_dir, f"{run_name}_KNe_single_model_summary.csv"))
all_table.to_csv(os.path.join(data_dir, f"{run_name}_KNe_all_models_summary.csv"))

headline_single = single_table.loc["multi_detect", "Fraction detected of total (mean)"]
headline_all = all_table.loc["multi_detect", "Fraction detected of total (mean)"]
print(
    f"GW170817-like KNe detection efficiency (multi_detect, of the full {n_events}-event population): "
    f"{headline_single:.1%}"
)
print(
    f"Population-averaged (all Bulla models) KNe detection efficiency (multi_detect): " f"{headline_all:.1%}"
)

## 7. Sky maps and histograms

`generate_kn_pop_slicer` returns a `UserPointsSlicer` (scattered injected events, not a fixed Healpix grid), so its native plotters are `BaseSkyMap` and `BaseHistogram` rather than the Healpix versions used in notebooks 01-03 - `bundle.plot()` still gives the equivalent "sky map + histogram" pair natively.

In [ ]:
def save_bundle_plots(bundle, tag, figs_dir):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
print("--- Single model: multi_detect ---")
saved = save_bundle_plots(
    bgroup_single.bundle_dict["KNePopMetric_single_multi_detect"],
    f"{run_name}_KNePopMetric_single_multi_detect",
    figs_dir,
)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = bgroup_single.bundle_dict["KNePopMetric_single_multi_detect"].plot(savefig=False)
plt.show()

In [ ]:
print("--- All models: multi_detect ---")
saved = save_bundle_plots(
    bgroup_all.bundle_dict["KNePopMetric_all_multi_detect"],
    f"{run_name}_KNePopMetric_all_multi_detect",
    figs_dir,
)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = bgroup_all.bundle_dict["KNePopMetric_all_multi_detect"].plot(savefig=False)
plt.show()

## 8. Detection efficiency across criteria: single model vs full population

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(criteria))
width = 0.35

single_eff = single_table.loc[criteria, "Fraction detected of total (mean)"] * 100
all_eff = all_table.loc[criteria, "Fraction detected of total (mean)"] * 100

ax.bar(x - width / 2, single_eff, width, label="single model (GW170817-like)", color="steelblue")
ax.bar(x + width / 2, all_eff, width, label="all models (full Bulla grid)", color="indianred")
ax.set_xticks(x)
ax.set_xticklabels(criteria, rotation=30, ha="right")
ax.set_ylabel("Detection efficiency [%]\n(fraction of injected events)")
ax.set_title(f"KNe detection efficiency by criterion - {run_name}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_KNe_efficiency_by_criterion")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 9. Caveats

- These are **detection efficiencies**, not expected event counts: turning a fraction into "expected KNe detections per year" requires folding in an actual binary-neutron-star/NS-BH merger volumetric rate density, which is not part of this metric. The `d_min=10, d_max=600` Mpc range and the (approximately uniform-in-volume) distance sampling in `generate_kn_pop_slicer` set the *injected* population, not an astrophysical rate.
- With `n_events` injected events, efficiency estimates carry Monte Carlo (binomial) sampling noise of order `1/sqrt(n_events)` - at `n_events=500000` this is small (~0.1%), but drops quickly if you reduce `n_events` for a faster test run.
- The `"all models"` run averages uniformly over the *entire* Bulla template grid (all ejecta masses, opening angles, and viewing angles with equal weight), which is not necessarily how real kilonovae are distributed in nature; the `"single model"` (GW170817-like) run instead isolates one specific, observationally anchored scenario.
- A more sophisticated companion metric, `PrestoColorKNePopMetric` (`rubin_sim.maf.maf_contrib.presto_color_kne_pop_metric`), additionally models photometric *classification* probability (not just detection) using a machine-learning color classifier; it is used by the official batch's "Presto KNe" subgroup and could be the subject of a future `04b` notebook, following the same pattern as `01b` for the 3x2pt FoM.
- As with `SNNSNMetric` (notebook 03), this metric relies on external template files (`$RUBIN_SIM_DATA_DIR/maf/bns/*.dat`) rather than the depth/visit-counting logic used in notebooks 01/01b/02; make sure `rs_download_data` has populated them before running Section 5.


## References
- Bulla, M. 2019, MNRAS 489, 5037, "POSSIS: predicting spectra, light curves and polarization for multi-dimensional models of supernovae and kilonovae" - defines the kilonova template grid used here.
- Andreoni, I., Coughlin, M. W. et al. 2021, ApJ 918, 63, "Fast-transient Searches in Real Time with ZTFReST" - origin of the `ztfrest_simple` rise/fade-rate detection criterion.
- Andrade, C. et al. 2025, PASP, "The Effect of Vera C. Rubin Observatory Cadence Selections on Kilonova Detectability" (arXiv:2502.14124) - a published cadence study using exactly `KNePopMetric`.
- Bianco, F. B. et al. 2022, ApJS 258, 1 - SCOC context.
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/maf_contrib/kne_metrics.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
